# Statnett mFRR EAM – Daily Bid Preparation

This notebook demonstrates a complete BSP bid-preparation workflow for Statnett (Norway)
using the `nexa_mfrr_eam` library.

## Workflow overview

1. Load day-ahead (DA) prices for the target trading day.
2. Load asset configuration (hydro, wind, battery).
3. Calculate bid prices accounting for **grunnrenteskatt (GS)** – the Norwegian ground
   rent tax on hydro and wind.
4. Build one `ReserveBid_MarketDocument` per asset containing 96 bids (one per MTU).
5. Validate each document against pre-MARI rules.
6. Preview the generated CIM XML for the first asset.
7. Summarise gate closure times for the first MTU.

## Grunnrenteskatt (GS tax)

Norway levies a resource rent tax on hydro power plants (37.6%) and offshore/onshore wind
farms (25%).  Because activation revenue is subject to GS, BSPs must **gross up** the price
they bid so that the after-tax revenue still covers operating costs and the target margin.

The formula is:

```
# For GS assets:
bid_price = DA_price + (overhead + profit_target) / (1 - gs_rate)

# For non-GS assets (batteries, etc.):
bid_price = DA_price + overhead + profit_target
```

This ensures that after paying the GS tax the BSP retains the required margin.

## 1. Imports

In [1]:
import json
from decimal import Decimal
from pathlib import Path

import pandas as pd

from nexa_mfrr_eam import (
    Bid,
    BidDocument,
    BiddingZone,
    MARIMode,
    MarketProductType,
    TSO,
)
from nexa_mfrr_eam.timing import gate_closure

DATA_DIR = Path("data")

## 2. Load Day-Ahead Prices

In [2]:
prices_df = pd.read_csv(
    DATA_DIR / "day_ahead_prices.csv",
    parse_dates=["mtu_start"],
    date_format="ISO8601",
)
# Ensure the column is timezone-aware UTC
prices_df["mtu_start"] = pd.to_datetime(prices_df["mtu_start"], utc=True)

print(f"Loaded {len(prices_df)} MTUs")
prices_df.head()

Loaded 96 MTUs


,mtu_start,da_price_eur_mwh
0,2026-03-21 00:00:00+00:00,41.93
1,2026-03-21 00:15:00+00:00,41.24
2,2026-03-21 00:30:00+00:00,40.77
3,2026-03-21 00:45:00+00:00,40.53
4,2026-03-21 01:00:00+00:00,40.54


In [3]:
prices_df["da_price_eur_mwh"].describe().round(2)

count    96.00
mean     62.63
std      11.14
min      40.53
25%      54.99
50%      63.30
75%      72.88
max      79.15
Name: da_price_eur_mwh, dtype: float64

## 3. Load Asset Configuration

In [4]:
assets = []
for asset_file in sorted(DATA_DIR.glob("asset_*.json")):
    with asset_file.open() as fh:
        assets.append(json.load(fh))

assets_df = pd.DataFrame(assets)
assets_df[["name", "resource_id", "volume_mw", "product_type", "gs_rate"]]

,name,resource_id,volume_mw,product_type,gs_rate
0,Battery Storage - Kvinesdal,NOKG90903,20,SCHEDULED_AND_DIRECT,0.000
1,Hydro Reservoir - Kvennfoss,NOKG90901,50,SCHEDULED_AND_DIRECT,0.376
2,Offshore Wind Farm - Utsira Nord,NOKG90902,30,SCHEDULED_ONLY,0.250


## 4. GS Tax Calculation

The `calculate_bid_price` function adds the required cost recovery margin on
top of the DA price, grossed up for GS tax where applicable.

In [5]:
def calculate_bid_price(
    da_price: float,
    overhead: float,
    profit_target: float,
    gs_rate: float,
) -> float:
    """Calculate the bid price accounting for grunnrenteskatt.

    For GS assets the required margin is divided by (1 - gs_rate) so that
    after paying the tax the BSP retains the full overhead and profit target.

    Args:
        da_price: Day-ahead market price in EUR/MWh.
        overhead: Operating cost in EUR/MWh.
        profit_target: Required profit margin in EUR/MWh.
        gs_rate: Grunnrenteskatt rate (0.0 = no tax, 0.376 = hydro, 0.25 = wind).

    Returns:
        Bid price in EUR/MWh, rounded to 2 decimal places.
    """
    if gs_rate > 0:
        return round(da_price + (overhead + profit_target) / (1 - gs_rate), 2)
    return round(da_price + overhead + profit_target, 2)


# Worked example
example_rows = []
for asset in assets:
    sample_da = 65.0
    bid_px = calculate_bid_price(
        sample_da,
        asset["overhead_eur_mwh"],
        asset["profit_target_eur_mwh"],
        asset["gs_rate"],
    )
    example_rows.append(
        {
            "Asset": asset["name"],
            "DA price": sample_da,
            "GS rate": asset["gs_rate"],
            "Overhead": asset["overhead_eur_mwh"],
            "Profit target": asset["profit_target_eur_mwh"],
            "Bid price": bid_px,
        }
    )
pd.DataFrame(example_rows)

,Asset,DA price,GS rate,Overhead,Profit target,Bid price
0,Battery Storage - Kvinesdal,65.0,0.000,3.2,8.0,76.20
1,Hydro Reservoir - Kvennfoss,65.0,0.376,2.5,5.0,77.02
2,Offshore Wind Farm - Utsira Nord,65.0,0.250,1.8,4.0,72.73


## 5. Build Bids

For each asset we build one bid per MTU (96 bids for the day).  Each bid is
a divisible up-regulation bid with the price grossed up for GS.

In [6]:
asset_bids: dict[str, list] = {}

for asset in assets:
    bids = []
    for _, row in prices_df.iterrows():
        bid_price = calculate_bid_price(
            float(row["da_price_eur_mwh"]),
            asset["overhead_eur_mwh"],
            asset["profit_target_eur_mwh"],
            asset["gs_rate"],
        )
        # row["mtu_start"] is a pandas Timestamp with UTC tzinfo
        mtu_iso = row["mtu_start"].isoformat()  # e.g. '2026-03-21T00:00:00+00:00'
        bid = (
            Bid.up(volume_mw=asset["volume_mw"], price_eur=bid_price)
            .divisible(min_volume_mw=asset["min_volume_mw"])
            .for_mtu(mtu_iso)
            .bidding_zone(BiddingZone[asset["bidding_zone"]])
            .resource(asset["resource_id"], coding_scheme=asset["coding_scheme"])
            .product_type(MarketProductType[asset["product_type"]])
            .build()
        )
        bids.append(bid)
    asset_bids[asset["name"]] = bids
    print(f"{asset['name']}: {len(bids)} bids built")

Battery Storage - Kvinesdal: 96 bids built
Hydro Reservoir - Kvennfoss: 96 bids built
Offshore Wind Farm - Utsira Nord: 96 bids built


## 6. Create Bid Documents

Each asset gets its own `ReserveBid_MarketDocument` targeting Statnett.

In [7]:
documents = {}

for asset in assets:
    doc = (
        BidDocument(tso=TSO.STATNETT)
        .sender(party_id="9999909919920", coding_scheme="A10")
        .add_bids(asset_bids[asset["name"]])
        .build()
    )
    documents[asset["name"]] = doc
    print(f"{asset['name']}: document created with {doc.time_series_count} time series")

Battery Storage - Kvinesdal: document created with 96 time series
Hydro Reservoir - Kvennfoss: document created with 96 time series
Offshore Wind Farm - Utsira Nord: document created with 96 time series


## 7. Validate Documents

In [8]:
all_valid = True
for asset_name, doc in documents.items():
    errors = doc.validate(mari_mode=MARIMode.PRE_MARI)
    if errors:
        all_valid = False
        print(f"ERRORS for {asset_name}:")
        for e in errors:
            print(f"  - {e}")
    else:
        print(f"OK: {asset_name}")

if all_valid:
    print("\nAll documents are valid!")

OK: Battery Storage - Kvinesdal
OK: Hydro Reservoir - Kvennfoss
OK: Offshore Wind Farm - Utsira Nord

All documents are valid!


## 8. Preview XML

Show the first 2000 characters of the generated CIM XML for the hydro asset.

In [9]:
first_doc = documents[assets[0]["name"]]
xml_bytes = first_doc.to_xml()
xml_preview = xml_bytes.decode("utf-8")[:2000]
print(xml_preview)

<?xml version='1.0' encoding='UTF-8'?>
<ReserveBid_MarketDocument xmlns="urn:iec62325.351:tc57wg16:451-7:reservebiddocument:7:4">
  <mRID>3f82b802-acf5-4256-a06a-5f46ef1e32b2</mRID>
  <revisionNumber>1</revisionNumber>
  <type>A37</type>
  <process.processType>A47</process.processType>
  <sender_MarketParticipant.mRID codingScheme="A10">9999909919920</sender_MarketParticipant.mRID>
  <sender_MarketParticipant.marketRole.type>A46</sender_MarketParticipant.marketRole.type>
  <receiver_MarketParticipant.mRID codingScheme="A01">10X1001A1001A38Y</receiver_MarketParticipant.mRID>
  <receiver_MarketParticipant.marketRole.type>A34</receiver_MarketParticipant.marketRole.type>
  <createdDateTime>2026-03-25T09:41:01Z</createdDateTime>
  <reserveBid_Period.timeInterval>
    <start>2026-03-21T00:00Z</start>
    <end>2026-03-22T00:00Z</end>
  </reserveBid_Period.timeInterval>
  <domain.mRID codingScheme="A01">10YNO-0--------C</domain.mRID>
  <subject_MarketParticipant.mRID codingScheme="A10">9999909

## 9. Summary

In [10]:
total_bids = sum(doc.time_series_count for doc in documents.values())
print(f"Total bids across all assets: {total_bids}")
print(f"Documents: {len(documents)}")

# Gate closure for the first MTU of the day
import datetime

first_mtu_start = datetime.datetime(2026, 3, 21, 0, 0, tzinfo=datetime.timezone.utc)
gc = gate_closure(first_mtu_start, mari_mode=MARIMode.PRE_MARI)
print(f"\nGate closure for first MTU ({first_mtu_start.isoformat()}):")
print(f"  BSP GCT (gate closes):     {gc.bsp_gct.isoformat()}")
print(f"  TSO GCT:                   {gc.tso_gct.isoformat()}")
print(f"  AOF run:                   {gc.aof_run.isoformat()}")
print(f"  Activation orders at:      {gc.activation.isoformat()}")
print(f"  Gate currently open:       {gc.is_gate_open()}")

Total bids across all assets: 288
Documents: 3

Gate closure for first MTU (2026-03-21T00:00:00+00:00):
  BSP GCT (gate closes):     2026-03-20T23:15:00+00:00
  TSO GCT:                   2026-03-20T23:45:00+00:00
  AOF run:                   2026-03-20T23:46:00+00:00
  Activation orders at:      2026-03-20T23:52:30+00:00
  Gate currently open:       False
